## This notebook presents the result analysis and visualization for submission NCOMMS-26-018997-T

**Title:** *Impacts and Benefits of Electrified Space Heating on Renewable-Based Power Systems*

This notebook contains the analysis of simulation results and the corresponding visualizations prepared for the above submission.

Extended Data Figure 5: Breakdown of major power system cost components, including operational, investment,
and trading costs.


**Contact:** Yi Guo  
**Email:** yi.guo@bit.edu.cn

### Extended Data Figure 5: cost and revenue breakdown across scenarios

This section loads the aggregated system cost data for four case pairs and compares the major cost and revenue categories between scenarios without and with heat pump (HP) flexibility.


In [ ]:
# =====================================================================
# This cell analyses and visualizes the results as shown in Extended Data Figure 5
# =====================================================================

import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from matplotlib.ticker import FuncFormatter
from matplotlib.patches import Patch
import matplotlib as mpl
from pathlib import Path
import os

# =========================
# Basic configuration
# =========================
ROOT = r'..\Output_data\power systems results'
CSV_TAIL = r'national_generation_and_capacity\0-systemcosts_percosttype_5_an.csv'


# Each tuple compares one scenario without HP flexibility (first) against
# its corresponding scenario with HP flexibility (second).
case_pairs = [
    ('case1a', 'case1a0'),
    ('case1b', 'case1b0'),
    ('case1c', 'case1c0'),
    ('case1d', 'case1d0'),
]

# Target years included in this figure.
data_cols = ['2030', '2040', '2050']
investment_labels = [f"Investment {y}" for y in data_cols]

# Morandi-style color palette for investment, operation, and trading components.
MORANDI = {
    'investment_2030': '#66c2a5',  # Green tone
    'investment_2040': '#fc8d62',  # Orange tone
    'investment_2050': '#ffd92f',  # Golden yellow
    'Variable Operation': '#e78ac3',  # Pink tone
    'Fixed Operation':    '#a6d854',  # Yellow-green tone
    'Trading':            '#8da0cb',  # Blue-purple tone
}

EDGE_COLOR = 'black'
LINEWIDTH = 2
FONT_BASE = 12
FIGSIZE = (16, 18)  # Four vertically stacked panels

BAR_WIDTH = 0.22
GROUP_GAP = 0.80      # Gap between year blocks
PAIR_GAP  = 0.0       # No gap between hatched and solid bars of the same color
COLOR_GAP = 0.30      # Gap between different cost/revenue categories within a year

# Y-axis unit scaling (billion euros).
YSCALE = 1e9
YUNIT  = '€ (B)'

# Label and connector placement settings.
TEXT_Y_OFFSET_POS = 0.02
TEXT_Y_OFFSET_NEG = 0.02
CONNECTOR_LINEWIDTH = 1.2

# Share the y-axis range across all panels for direct comparison.
SHARE_Y = True

# Asymmetric top and bottom padding to improve visibility of negative values.
BOTTOM_PAD_NEG = 0.35
TOP_PAD_POS    = 0.25

# Font sizes for axis ticks and labels.
AXIS_TICK_SIZE = 16
Y_LABEL_SIZE   = 16

# =========================
# Helper functions
# =========================
def fmt_billions(x, pos):
    """Format numeric axis values in billions."""
    return f"{x:.1f}"


def bar_color_for_other(label):
    """Return the predefined color for non-investment categories."""
    return MORANDI.get(label, '#A9A9A9')


def load_case_pair(case, case0):
    """Load and organize investment and non-investment data for a case pair."""
    f  = fr'{ROOT}\{case}\{CSV_TAIL}'
    f0 = fr'{ROOT}\{case0}\{CSV_TAIL}'

    f = os.path.abspath(f)
    f0 = os.path.abspath(f0)

    raw_df  = pd.read_csv(f)
    raw_df0 = pd.read_csv(f0)

    inv_df   = raw_df[raw_df['Row'].str.startswith('Investments')].copy()
    inv_df0  = raw_df0[raw_df0['Row'].str.startswith('Investments')].copy()
    other_df = raw_df[~raw_df['Row'].str.startswith('Investments') & ~raw_df['Row'].str.startswith('Transmission')].copy()
    other_df0= raw_df0[~raw_df0['Row'].str.startswith('Investments') & ~raw_df0['Row'].str.startswith('Transmission')].copy()

    investments_df  = pd.DataFrame(index=investment_labels, columns=data_cols).fillna(0)
    investments_df0 = pd.DataFrame(index=investment_labels, columns=data_cols).fillna(0)

    # Aggregate investment entries by decision year.
    for _, row in inv_df.iterrows():
        decision_year = row['Row'].split()[-1]
        key = f"Investment {decision_year}"
        if key in investment_labels:
            for col in data_cols:
                investments_df.loc[key, col] += row[col]

    for _, row in inv_df0.iterrows():
        decision_year = row['Row'].split()[-1]
        key = f"Investment {decision_year}"
        if key in investment_labels:
            for col in data_cols:
                investments_df0.loc[key, col] += row[col]

    o_df  = other_df.set_index('Row')
    o_df0 = other_df0.set_index('Row')
    other_labels = list(o_df.index)
    return investments_df, investments_df0, o_df, o_df0, other_labels


def draw_diff_label_and_connector(ax, x_center, base_val, main_val, yrange):
    """Annotate the relative and absolute difference between paired bars."""
    if base_val == 0 or np.isnan(base_val) or np.isnan(main_val):
        return
    diff = main_val - base_val
    rel = diff / base_val * 100

    if max(base_val, main_val) >= 0 and min(base_val, main_val) >= 0:
        y_text = max(base_val, main_val) + yrange * TEXT_Y_OFFSET_POS
        va = 'bottom'
    elif max(base_val, main_val) <= 0 and min(base_val, main_val) <= 0:
        y_text = min(base_val, main_val) - yrange * TEXT_Y_OFFSET_NEG
        va = 'top'
    else:
        side_val = main_val if abs(main_val) >= abs(base_val) else base_val
        y_text = side_val + yrange * (TEXT_Y_OFFSET_POS if side_val >= 0 else -TEXT_Y_OFFSET_NEG)
        va = 'bottom' if side_val >= 0 else 'top'

    ax.text(x_center, y_text,
            f"{rel:.3f} %\n({diff/1e6:.3f} M)",
            ha='center', va=va, fontsize=FONT_BASE, fontweight='bold')
    ax.plot([x_center - BAR_WIDTH/2 + PAIR_GAP/2, x_center + BAR_WIDTH/2 - PAIR_GAP/2],
            [base_val, main_val], linewidth=CONNECTOR_LINEWIDTH, color='black')


def apply_asymmetric_padding(ax):
    """Apply asymmetric vertical padding to improve the balance of the plot."""
    ymin, ymax = ax.get_ylim()
    span = ymax - ymin if ymax > ymin else 1.0
    ax.set_ylim(ymin - BOTTOM_PAD_NEG*span, ymax + TOP_PAD_POS*span)


def plot_one(ax, case, case0, investments_df, investments_df0, o_df, o_df0,
             other_labels, fixed_ylim=None, panel_label=None):
    """Plot one panel comparing a baseline case against its flexible counterpart."""
    n_years = len(data_cols)
    n_groups = 1 + len(other_labels)

    # Horizontal span of one color group: two adjacent bars plus the gap to the next group.
    color_block = (BAR_WIDTH * 2 + PAIR_GAP)
    unit_span   = color_block + COLOR_GAP
    group_block = unit_span * n_groups + GROUP_GAP

    base_x = np.arange(n_years) * group_block

    # Plot stacked investment bars as the first color group.
    inv_x_left  = base_x
    inv_x_right = inv_x_left + BAR_WIDTH + PAIR_GAP

    bottom0 = np.zeros(n_years)
    bottom  = np.zeros(n_years)
    inv_colors = [MORANDI['investment_2030'], MORANDI['investment_2040'], MORANDI['investment_2050']]

    # Baseline stacked investments without hatching.
    for i, label in enumerate(investments_df.index):
        vals0 = investments_df0.loc[label, data_cols].values
        ax.bar(inv_x_left, vals0, BAR_WIDTH, bottom=bottom0, color=inv_colors[i],
               edgecolor=EDGE_COLOR, linewidth=LINEWIDTH)
        bottom0 += vals0

    # Flexible-case stacked investments with hatching.
    for i, label in enumerate(investments_df.index):
        vals = investments_df.loc[label, data_cols].values
        ax.bar(inv_x_right, vals, BAR_WIDTH, bottom=bottom, color=inv_colors[i],
               edgecolor=EDGE_COLOR, linewidth=LINEWIDTH, hatch='///')
        bottom += vals

    # Plot all remaining categories as paired adjacent bars.
    for j, label in enumerate(other_labels):
        color = bar_color_for_other(label)
        base_pos = base_x + unit_span * (j + 1)
        main_pos = base_pos + BAR_WIDTH + PAIR_GAP
        vals0 = o_df0.loc[label, data_cols].values
        vals  = o_df.loc[label,  data_cols].values

        # Baseline bars without hatching.
        ax.bar(base_pos, vals0, BAR_WIDTH, color=color, edgecolor=EDGE_COLOR, linewidth=LINEWIDTH)
        # Flexible-case bars with hatching.
        ax.bar(main_pos, vals,  BAR_WIDTH, color=color, edgecolor=EDGE_COLOR, linewidth=LINEWIDTH, hatch='///')

    if fixed_ylim is not None:
        ax.set_ylim(*fixed_ylim)

    fig = ax.figure
    fig.canvas.draw()
    ymin, ymax = ax.get_ylim()
    yrange = ymax - ymin

    # Add difference connectors and annotations for total investment values.
    inv_centers = (inv_x_left + inv_x_right) / 2
    total0 = investments_df0[data_cols].values.sum(axis=0)
    total  = investments_df[data_cols].values.sum(axis=0)
    for xc, b0, b1 in zip(inv_centers, total0, total):
        draw_diff_label_and_connector(ax, xc, b0, b1, yrange)

    # Add difference connectors and annotations for the other categories.
    for j, label in enumerate(other_labels):
        base_pos = base_x + unit_span * (j + 1)
        main_pos = base_pos + BAR_WIDTH + PAIR_GAP
        centers  = (base_pos + main_pos) / 2
        vals0 = o_df0.loc[label, data_cols].values
        vals  = o_df.loc[label,  data_cols].values
        for xc, v0, v1 in zip(centers, vals0, vals):
            draw_diff_label_and_connector(ax, xc, v0, v1, yrange)

    if fixed_ylim is None:
        apply_asymmetric_padding(ax)

    # Place x-axis ticks at the center of each year block.
    group_centers = inv_centers + unit_span * (len(other_labels)) / 2
    ax.set_xticks(group_centers)
    ax.set_xticklabels(data_cols, fontsize=AXIS_TICK_SIZE)
    ax.set_xlabel('')

    # Format the y-axis in billions.
    ax.yaxis.set_major_formatter(FuncFormatter(lambda v, pos: fmt_billions(v / YSCALE, pos)))
    ax.tick_params(axis='y', labelsize=AXIS_TICK_SIZE, width=1.5)

    # Add a horizontal zero line to separate positive and negative values.
    ax.axhline(0, color='black', linewidth=1.0, alpha=0.6, zorder=0)

    # Draw dashed separators between year blocks.
    if len(group_centers) >= 2:
        midpoints = (group_centers[:-1] + group_centers[1:]) / 2.0
        for mp in midpoints:
            ax.axvline(mp, color='gray', linestyle='--', linewidth=0.5)

    # Optionally add panel labels such as a, b, c, and d.
    if panel_label:
        ax.text(0.01, 0.98, panel_label, transform=ax.transAxes, ha='left', va='top',
                fontsize=14, fontweight='bold')

    # Thicken all plot spines for publication-quality appearance.
    for spine in ax.spines.values():
        spine.set_linewidth(1.5)


def compute_global_ylim(all_data):
    """Compute a shared y-axis range across all subplots."""
    ymin, ymax = 0.0, 0.0
    for investments_df, investments_df0, o_df, o_df0, _ in all_data:
        tot0 = investments_df0[data_cols].sum(axis=0).values
        tot  = investments_df[data_cols].sum(axis=0).values
        arrs = [tot0, tot]
        for lbl in o_df.index:
            arrs += [o_df.loc[lbl, data_cols].values, o_df0.loc[lbl, data_cols].values]
        vals = np.concatenate(arrs)
        ymin = min(ymin, np.nanmin(vals))
        ymax = max(ymax, np.nanmax(vals))
    span = ymax - ymin if ymax > ymin else (abs(ymax) + 1.0)
    return (ymin - BOTTOM_PAD_NEG*span, ymax + TOP_PAD_POS*span)


# =====================================================================
# Data loading and plotting
# =====================================================================
all_case_data = [load_case_pair(c, c0) for c, c0 in case_pairs]
fixed_ylim = compute_global_ylim(all_case_data) if SHARE_Y else None

fig, axes = plt.subplots(4, 1, figsize=FIGSIZE, sharex=False, sharey=SHARE_Y)

panel_labels = ['a', 'b', 'c', 'd']
for idx, ((case, case0), ax) in enumerate(zip(case_pairs, axes)):
    investments_df, investments_df0, o_df, o_df0, other_labels = all_case_data[idx]
    # Pass panel_label=panel_labels[idx] if panel labels should be displayed.
    plot_one(ax, case, case0, investments_df, investments_df0, o_df, o_df0,
             other_labels, fixed_ylim=fixed_ylim, panel_label=None)

# Add a shared legend below the figure.
handles = [
    Patch(facecolor='white', edgecolor=EDGE_COLOR, label='without HP flexibility'),
    Patch(facecolor='white', hatch='///', edgecolor=EDGE_COLOR, label='with HP flexibility'),
    Patch(facecolor=MORANDI['investment_2030'], edgecolor=EDGE_COLOR, label='Investment 2030'),
    Patch(facecolor=MORANDI['investment_2040'], edgecolor=EDGE_COLOR, label='Investment 2040'),
    Patch(facecolor=MORANDI['investment_2050'], edgecolor=EDGE_COLOR, label='Investment 2050'),
    Patch(facecolor=MORANDI['Variable Operation'], edgecolor=EDGE_COLOR, label='Variable operation'),
    Patch(facecolor=MORANDI['Fixed Operation'], edgecolor=EDGE_COLOR, label='Fixed operation'),
    Patch(facecolor=MORANDI['Trading'], edgecolor=EDGE_COLOR, label='Trading'),
]
fig.legend(handles=handles, loc='lower center', ncol=4, fontsize=16, frameon=False)

# Add a shared y-axis title.
fig.text(-0.02, 0.5, f'Cost / Revenue, {YUNIT}', va='center', rotation='vertical',
         fontsize=Y_LABEL_SIZE)

# Apply a fixed y-axis range across all subplots if desired.
for ax in axes:
    ax.set_ylim(-1e9, 2e9)

plt.tight_layout(rect=[0, 0.05, 1, 1])
plt.subplots_adjust(hspace=0.2)

# Optional transparent background settings.
plt.rcParams['figure.facecolor'] = 'none'
plt.rcParams['axes.facecolor'] = 'none'

# ======================= SVG export added for Adobe Illustrator editing =======================
# Keep SVG text as editable text objects rather than converting them to paths.
mpl.rcParams['svg.fonttype'] = 'none'
mpl.rcParams['text.usetex'] = False
mpl.rcParams['font.family'] = ['Arial']

# # Output path for the SVG file.
# out_path = Path.cwd() / 'figure_for_AI.svg'

# # Save the figure as SVG with transparent background and tight margins.
# fig.savefig(
#     out_path,
#     format='svg',
#     bbox_inches='tight',
#     pad_inches=0.02,
#     facecolor='none',
#     edgecolor='none',
#     transparent=True
# )
# print(f'Saved SVG to: {out_path}')
# ==============================================================================================

plt.show()



C:\Users\Administrator\AppData\Local\Temp\ipykernel_42308\2072708610.py:103: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  investments_df  = pd.DataFrame(index=investment_labels, columns=data_cols).fillna(0)
C:\Users\Administrator\AppData\Local\Temp\ipykernel_42308\2072708610.py:104: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  investments_df0 = pd.DataFrame(index=investment_labels, columns=data_cols).fillna(0)
C:\Users\Administrator\AppData\Local\Temp\ipykernel_42308\2072708610.py:112: FutureWarning: Setting an item of incompatible dtype is deprecated 